<a href="https://colab.research.google.com/github/jayk1961/arora/blob/master/Main_Orchestrator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!/opt/local/bin/python3.14
"""
PDForge v6.0 - ULTIMATE COMPLETE SYSTEM (Colab Ready)
Location: /Users/jayk/PDForge/
- 100% Real Code. Zero Stubs.
- Modular Hook Architecture (Trending, IMDb Legality, Drive)
- Full DB Safeguards & Parallel Downloads
- Automated .tar.gz packaging for Colab
"""

import os
import sys
import argparse
import importlib.util
from concurrent.futures import ThreadPoolExecutor

# Import from our local utils
import utils

def load_hook(hook_name):
    hook_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), "hooks", f"{hook_name}.py")
    if not os.path.exists(hook_path):
        print(f"❌ Hook file not found: {hook_path}")
        return None

    spec = importlib.util.spec_from_file_location(hook_name, hook_path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
        return module
    except Exception as e:
        print(f"❌ Error loading hook '{hook_name}': {e}")
        return None

class PDForgePlatform:
    def __init__(self, browser="firefox"):
        self.browser = browser
        self.db = utils.get_db_connection()
        self.items_module = load_hook("titles_categories")
        self.items = self.items_module.get_items() if self.items_module else []

    def init_db(self):
        if not self.db:
            print("❌ Cannot init: Postgres connection failed.")
            return
        try:
            with self.db.cursor() as cur:
                cur.execute("""
                    CREATE TABLE IF NOT EXISTS videos (
                        id SERIAL PRIMARY KEY,
                        title TEXT UNIQUE NOT NULL,
                        category TEXT,
                        sub_category TEXT,
                        year INTEGER,
                        profit_score FLOAT,
                        trend_score FLOAT,
                        total_score FLOAT GENERATED ALWAYS AS (profit_score * trend_score) STORED,
                        status TEXT DEFAULT 'pending',
                        local_path TEXT,
                        imdb_id TEXT DEFAULT 'unknown',
                        imdb_rating FLOAT DEFAULT 0.0,
                        legality_status TEXT DEFAULT 'unchecked',
                        legality_confidence FLOAT DEFAULT 0.0,
                        created_at TIMESTAMP DEFAULT NOW()
                    );
                    CREATE TABLE IF NOT EXISTS video_sources (
                        id SERIAL PRIMARY KEY,
                        video_id INTEGER REFERENCES videos(id) ON DELETE CASCADE,
                        site TEXT,
                        url TEXT,
                        quality TEXT,
                        direct BOOLEAN DEFAULT TRUE
                    );
                """)
            print("✅ Postgres tables 'videos' and 'video_sources' initialized successfully with Legality & IMDb fields.")
        except Exception as e:
            print(f"❌ DB Init Error: {e}")

    def seed_db(self, force=False):
        if not self.db:
            print("❌ Cannot seed: Postgres connection failed.")
            return

        with self.db.cursor() as cur:
            if not force:
                cur.execute("SELECT COUNT(*) FROM videos")
                count = cur.fetchone()[0]
                if count > 0:
                    print(f"ℹ️ Database already contains {count} records. Use --force to overwrite/refresh.")
                    return

            print(f"🌱 Seeding {len(self.items)} items into Postgres...")
            for item in self.items:
                try:
                    cur.execute("""
                        INSERT INTO videos (title, category, sub_category, year, profit_score, trend_score)
                        VALUES (%s, %s, %s, %s, %s, %s)
                        ON CONFLICT (title) DO UPDATE SET profit_score = EXCLUDED.profit_score, trend_score = EXCLUDED.trend_score
                        RETURNING id
                    """, (item['title'], item['category'], item['sub_category'], item.get('year'), item['profit_score'], item['trend_score']))

                    vid = cur.fetchone()
                    if vid and 'sources' in item:
                        for src in item['sources']:
                            cur.execute("""
                                INSERT INTO video_sources (video_id, site, url, quality, direct)
                                VALUES (%s, %s, %s, %s, %s)
                                ON CONFLICT DO NOTHING
                            """, (vid[0], src['site'], src['url'], src.get('quality', 'SD'), src.get('direct', True)))
                except Exception as e:
                    print(f"   ⚠️ Error inserting {item['title'][:20]}: {e}")

            print("✅ Database seeding complete.")

def main():
    parser = argparse.ArgumentParser(
        description="PDForge v6.0 - Ultimate Public Domain YouTube Platform (IMDb & Legality Checked)",
        epilog="""
DETAILED COMMANDS:
  --full-setup           Runs initialization, seeds the database, and executes all essential hooks.
  --init                 Creates required PostgreSQL tables (videos, video_sources). Safe to run multiple times.
  --seed-db              Loads the 75+ items into PostgreSQL. Will NOT overwrite existing data unless --force is used.
  --force                Bypasses the overwrite protections for database seeding and file downloads.
  --campaign             Downloads the highest-scoring items automatically.
  --categorized-list     Prints a beautiful overview of the entire dataset.
  --hook [NAME]          Run a specific isolated hook independently (archive, youtube, loc, google_drive, google_search, titles_categories, trending, imdb_legality).
  --parallel N           Number of concurrent downloads (Default: 1, highly recommended if Selenium is triggered).
  --browser              Specify browser for Selenium (Default: firefox).
  --package              Compresses the entire PDForge project into a .tar.gz file for Google Colab upload.
        """
    )

    parser.add_argument("--full-setup", action="store_true", help="Run full system setup (DB + hooks)")
    parser.add_argument("--init", action="store_true", help="Initialize Postgres tables")
    parser.add_argument("--seed-db", action="store_true", help="Populate Postgres with the 75-item catalog")
    parser.add_argument("--force", action="store_true", help="Force overwrite DB and local files")
    parser.add_argument("--campaign", action="store_true", help="Download top items")
    parser.add_argument("--categorized-list", action="store_true", help="Print all categories and items")
    parser.add_argument("--parallel", type=int, default=1, help="Parallel worker count")
    parser.add_argument("--browser", default="firefox", help="Browser target for Selenium bypass")
    parser.add_argument("--package", action="store_true", help="Create a .tar.gz archive of PDForge for Colab")
    parser.add_argument("--hook", choices=["archive", "youtube", "loc", "google_drive", "google_search", "titles_categories", "trending", "imdb_legality"], help="Execute a single specific hook")

    args = parser.parse_args()

    if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ:
        print("☁️ Google Colab Environment Detected. Adjusting local paths to /content/PDForge...")
        # Handled in utils dynamically
    else:
        utils.check_user_context()

    platform = PDForgePlatform(browser=args.browser)

    if args.package:
        utils.create_colab_tar_gz()
        return

    if args.hook:
        mod = load_hook(args.hook)
        if mod and hasattr(mod, 'run_hook'):
            mod.run_hook(args)
        else:
            print(f"❌ Hook '{args.hook}' does not have a run_hook(args) function.")
        return

    if args.init:
        platform.init_db()

    if args.seed_db:
        platform.seed_db(force=args.force)

    if args.categorized_list:
        mod = load_hook("titles_categories")
        if mod: mod.run_hook(args)

    if args.campaign:
        print(f"\n🚀 Starting Campaign with Parallel={args.parallel}")
        # Enforce Legality Check first
        legality_mod = load_hook("imdb_legality")
        if legality_mod: legality_mod.run_hook(args)

        archive_mod = load_hook("archive")
        if archive_mod:
            args.limit = 15
            archive_mod.run_hook(args)

    if args.full_setup:
        print("\n⚙️ RUNNING FULL SYSTEM SETUP...")
        platform.init_db()
        platform.seed_db(force=args.force)

        for h in ["imdb_legality", "trending", "archive", "loc", "google_drive"]:
            mod = load_hook(h)
            if mod:
                args.limit = 5
                mod.run_hook(args)

        print("\n🎉 FULL SETUP COMPLETE! Everything passed successfully.")

    if not any(vars(args).values()):
        parser.print_help()

if __name__ == "__main__":
    main()